# ResNet50 Transfer Learning Model
Same structure as VGG16 notebook — upgraded to ResNet50 🔥

In [ ]:
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## STEP 1 — TPU Setup

In [ ]:
import tensorflow as tf

try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    print("✅ TPU enabled")
except:
    strategy = tf.distribute.get_strategy()
    print("⚠️ TPU not found, using CPU/GPU")

print("Replicas:", strategy.num_replicas_in_sync)

## STEP 2 — PATHS
⚠️ ResNet50 bhi 224x224 accept karta hai — koi change nahi!

In [ ]:
TRAIN_DIR = '/kaggle/input/datasets/rajpoot75/psoriasis-dataset/hlo/train'
TEST_DIR = '/kaggle/input/datasets/rajpoot75/psoriasis-dataset/hlo/test'

IMG_SIZE   = (224, 224)   # ResNet50 ka standard size
BATCH_SIZE = 32 * strategy.num_replicas_in_sync

## STEP 3 — DATA AUGMENTATION
ResNet50 ka apna preprocess_input hai — VGG16 wala mat lagana!

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input  # ✅ ResNet50 specific

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input   # ✅ test mein bhi same
)

## STEP 4 — LOAD DATASET

In [ ]:
train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print("Classes:", train_gen.class_indices)
print("Train samples:", train_gen.samples)
print("Test samples:", test_gen.samples)

## STEP 5 — CLASS WEIGHTS

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

labels = train_gen.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

## STEP 6 — ResNet50 MODEL (Transfer Learning)

**ResNet50 vs VGG16 fark:**
- ResNet50 mein **Residual/Skip Connections** hain — vanishing gradient problem nahi hoti
- VGG16 se **halka** hai (25M params vs 138M) — faster training!
- Generally **VGG16 se zyada accurate** hota hai

**Strategy:**
- Phase 1: Base layers **freeze** → sirf top layers train (fast)
- Phase 2: Last ResNet block **unfreeze** → fine-tuning (accuracy boost)

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization

def build_resnet50_model(trainable_base=False):
    """
    trainable_base=False  → Phase 1: base frozen  (fast, stable)
    trainable_base=True   → Phase 2: fine-tuning  (higher accuracy)
    """
    # ResNet50 base — ImageNet weights, top classifier exclude
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,          # apna classifier lagayenge
        input_shape=(224, 224, 3)
    )

    # Phase 1: Poora base freeze
    base_model.trainable = False

    # Phase 2: sirf last ResNet block unfreeze (conv5_block1, 2, 3)
    if trainable_base:
        for layer in base_model.layers:
            layer.trainable = False
        # ResNet50 mein last block 'conv5' se shuru hota hai
        for layer in base_model.layers:
            if 'conv5' in layer.name:
                layer.trainable = True
        print("🔓 conv5 block unfrozen for fine-tuning")

    # ---- Custom Top Classifier ----
    x = base_model.output
    x = GlobalAveragePooling2D()(x)      # GAP — same as tera CNN
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)   # binary classification

    model = Model(inputs=base_model.input, outputs=output)
    return model

## STEP 7 — COMPILE (PHASE 1: Base Frozen)

In [ ]:
with strategy.scope():
    model = build_resnet50_model(trainable_base=False)   # Phase 1

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),        # Phase 1: higher LR theek hai
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall(),
            tf.keras.metrics.AUC()
        ]
    )

model.summary()

trainable = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f"\n✅ Trainable params: {trainable:,}  (sirf top layers)")

## STEP 8 — RESUME CHECK

In [ ]:
model_path   = "best_resnet50.keras"
history_path = "history_resnet_phase1.csv"

if os.path.exists(model_path):
    print("🔄 Resuming training...")
    model = tf.keras.models.load_model(model_path)
    if os.path.exists(history_path):
        df = pd.read_csv(history_path)
        initial_epoch = len(df)
    else:
        initial_epoch = 0
    print("Starting from epoch:", initial_epoch)
else:
    print("🚀 Fresh training...")
    initial_epoch = 0

## STEP 9 — CALLBACKS

In [ ]:
callbacks_phase1 = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_resnet50.keras",
        save_best_only=False,
        save_freq='epoch',
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        "history_resnet_phase1.csv",
        append=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        patience=7,
        restore_best_weights=True,
        mode='max',
        verbose=1
    )
]

## STEP 10 — TRAINING PHASE 1 (Frozen Base — 20 epochs)

In [ ]:
print("🚀 PHASE 1: Training top layers only (ResNet50 base frozen)")

history_p1 = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=20,
    initial_epoch=initial_epoch,
    class_weight=class_weights,
    callbacks=callbacks_phase1
)

print("✅ Phase 1 complete!")

## STEP 11 — FINE TUNING (Phase 2)

ResNet50 ka **conv5 block** unfreeze karenge — yeh last aur most powerful block hai!

⚠️ **Bahut chhoti LR use karo** (`1e-5`) — warna pretrained weights kharab ho jayenge!

In [ ]:
print("🔓 PHASE 2: Fine-tuning ResNet50 conv5 block...")

# Phase 1 ka best model load karo
model = tf.keras.models.load_model("best_resnet50.keras")

# Pehle sab freeze, phir conv5 unfreeze
for layer in model.layers:
    layer.trainable = False

for layer in model.layers:
    if hasattr(layer, 'name') and 'conv5' in layer.name:
        layer.trainable = True

# Top classifier bhi trainable rakho
for layer in model.layers[-6:]:
    layer.trainable = True

# Recompile — VERY low LR
with strategy.scope():
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),   # ⚠️ Bahut chhoti LR
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall(),
            tf.keras.metrics.AUC()
        ]
    )

trainable = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f"✅ Fine-tuning trainable params: {trainable:,}")

callbacks_phase2 = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_resnet50_finetuned.keras",
        save_best_only=False,
        save_freq='epoch',
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        "history_resnet_phase2.csv",
        append=False
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        patience=7,
        restore_best_weights=True,
        mode='max',
        verbose=1
    )
]

history_p2 = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=30,
    class_weight=class_weights,
    callbacks=callbacks_phase2
)

print("✅ Phase 2 Fine-tuning complete!")

## STEP 12 — EVALUATION

In [ ]:
loss, acc, prec, rec, auc = model.evaluate(test_gen)

print(f"\n{'='*40}")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"AUC       : {auc:.4f}")
print(f"{'='*40}")

## STEP 13 — CONFUSION MATRIX

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = model.predict(test_gen)
y_pred = (y_pred > 0.5).astype(int)

cm = confusion_matrix(test_gen.classes, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=list(train_gen.class_indices.keys()),
            yticklabels=list(train_gen.class_indices.keys()))
plt.title('Confusion Matrix — ResNet50')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print(classification_report(test_gen.classes, y_pred,
      target_names=list(train_gen.class_indices.keys())))

## STEP 14 — TRAINING CURVES 📈

In [ ]:
df_p1 = pd.read_csv("history_resnet_phase1.csv")
df_p2 = pd.read_csv("history_resnet_phase2.csv")
df_p2['epoch'] = df_p2['epoch'] + len(df_p1)

df_all = pd.concat([df_p1, df_p2], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_all['accuracy'], label='Train Accuracy')
axes[0].plot(df_all['val_accuracy'], label='Val Accuracy')
axes[0].axvline(len(df_p1)-1, color='red', linestyle='--', label='Fine-tune start')
axes[0].set_title('Accuracy — ResNet50')
axes[0].legend()

axes[1].plot(df_all['loss'], label='Train Loss')
axes[1].plot(df_all['val_loss'], label='Val Loss')
axes[1].axvline(len(df_p1)-1, color='red', linestyle='--', label='Fine-tune start')
axes[1].set_title('Loss — ResNet50')
axes[1].legend()

plt.suptitle('ResNet50 Training Curves', fontsize=14)
plt.tight_layout()
plt.show()

## STEP 15 — SAVE FINAL MODEL

In [ ]:
model.save("resnet50_final.keras")
print("✅ ResNet50 Model Saved Successfully!")